# Road Damage Severity Index (SI)
This notebook will demonstrate how to calculate a Road Severity Index using YOLOv8 inference outputs.

## 1. YOLOv8 Inference Output Structure
When we run `results = model.predict(image)`, YOLOv8 returns a list of `Results` objects one for each input image.

The `Results` object contains different attributes depending on the task. For object detection, the primary attribute is `results.boxes`, which is a specialized `Boxes` object containing all detected bounding boxes. 

Inside `results.boxes`, I can extract the following:
- `boxes.xyxy`: This bounds box coordinates in `[x1, y1, x2, y2]` format.
- `boxes.conf`: This gives the confidence score.
- `boxes.cls`: Basically gives the class ID for the detected object.

## 2. Severity Index Formula

To calculate a Severity Index (SI) that quantifies the overall damage on a road given a single frame, we need to balance the severity of the damage type, the model's confidence, and the physical footprint of the damage relative to the camera view.

**Weights ($W$)**
- Potholes: `1.0` indicates highest sevearity
- Alligator Cracks: `0.8` High sevearity
- Longitudinal Cracks: `0.5` Medium severity
- Transverse Cracks: `0.3` Low severity

**Formula**
The formula for severity index that I am going to be using is:
$$SI = \sum_{i=1}^{n} (W_i \times Confidence_i \times A_{rel, i})$$

Where $A_{rel}$ is the bounding box area relative to the total frame area.

In [2]:
import numpy as np
def calculate_severity_index(results, frame_area, weights=None):
    
    if weights is None:
        #Indexed as 0: Potholes, 1: Alligator, 2: Longitudinal, 3: Transverse
        weights = { 0: 1.0, 1: 0.8, 2: 0.5, 3: 0.3 }
        
    severity_index = 0.0
    
    # Checking for detection
    if results.boxes is None or len(results.boxes) == 0:
        return severity_index
        
    boxes = results.boxes
    
    # Array extraction
    coords = boxes.xyxy.cpu().numpy() 
    confidences = boxes.conf.cpu().numpy()
    class_ids = boxes.cls.cpu().numpy()
    
    for i in range(len(boxes)):
        x1, y1, x2, y2 = coords[i]
        
        # Bounding box area and relative area calculation
        bbox_area = (x2 - x1) * (y2 - y1)
        relative_area = bbox_area / frame_area
        
        # Finally calculate the Severity Index 
        weight = weights.get(int(class_ids[i]), 0.1)
        severity_index += weight * confidences[i] * relative_area
        
    return severity_index

In [ ]:
# Testing with actual training data
import os
import cv2
import glob
import numpy as np

# Get a sample image and label
train_images = glob.glob("../RDD_SPLIT/train/images/*.jpg")
if not train_images:
    print("No training images found!")
else:
    sample_img_path = train_images[0]
    sample_label_path = sample_img_path.replace("images", "labels").replace(".jpg", ".txt")
    
    img = cv2.imread(sample_img_path)
    h, w, _ = img.shape
    total_area = w * h
    
    print(f"Testing on image: {os.path.basename(sample_img_path)}")
    print(f"Frame Area: {total_area} px^2")
    
    # Read ground truth labels
    bboxes = []
    confidences = []
    class_ids = []
    
    if os.path.exists(sample_label_path):
        with open(sample_label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_c, y_c, bw, bh = map(float, parts[1:5])
                    
                    # Convert normalized YOLO format to xyxy
                    x1 = (x_c - bw / 2) * w
                    y1 = (y_c - bh / 2) * h
                    x2 = (x_c + bw / 2) * w
                    y2 = (y_c + bh / 2) * h
                    
                    bboxes.append([x1, y1, x2, y2])
                    confidences.append(1.0) 
                    class_ids.append(class_id)
    
    class MockBoxes:
        def __len__(self): return len(bboxes)
        
    class MockResults:
        pass
    
    results = MockResults()
    results.boxes = MockBoxes()
    
    if len(bboxes) > 0:
        results.boxes.xyxy = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array(bboxes)})()})()
        results.boxes.conf = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array(confidences)})()})()
        results.boxes.cls = type('MockTensor', (), {'cpu': lambda self: type('MockCPU', (), {'numpy': lambda self: np.array(class_ids)})()})()
    else:
        results.boxes = None
    
    print("\nExtraction Example:")
    if results.boxes is not None:
        for i in range(len(bboxes)):
            x1, y1, x2, y2 = bboxes[i]
            print(f"Detection {i+1}: BBox ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}), Class ID: {class_ids[i]}, Conf: {confidences[i]:.4f}")
    
    si = calculate_severity_index(results, total_area)
    print(f"\nCalculated Severity Index: {si:.5f}")


Testing on image: China_Drone_000001.jpg
Frame Area: 262144 px^2

Extraction Example:
Detection 1: BBox (14.0, 70.0, 198.0, 99.0), Class ID: 1, Conf: 1.0000
Detection 2: BBox (95.0, 114.0, 120.0, 272.0), Class ID: 0, Conf: 1.0000

Calculated Severity Index: 0.03135
